> ✅ **Completed by Ketan Khanna**

> Big Data Management Systems and Tools · University of Toronto School of Continuing Studies · 2021 · Assignment 1

## Assignment 1: MongoDB

This assignment is based on content discussed in Module 2: Introduction to MongoDB

## Learning outcomes 

The purpose of this assignment is for learners to be able to:
- Familarize with JSON document syntax
- Understand basic MongoDB CRUD operations
- Understand MongoDB data pipelines to run aggregate queries

In this assignment, you will make use of the sample data provided in Module 2.  

This dataset has 3 collections: Employee, Workplace and Address.  You will import this data into your local MongoDB database.

Required imports for this project are given below. Make sure you have all libraries required for this project installed. You may use conda or pip based on your set up.

## Setup Notes

**Please note** These instructions are duplicated here for your reference. You would have encountered them originally in Module 2 Part 2. It is expected that you have already taken the steps to set up and run MongoDB.

We will be using MongoDB Community Edition.  The MongoDB database <b>MUST</b> be installed and running locally before continuing with this notebook.  We will need to install two packages using the Anaconda package manager:

1. [Mongodb](https://www.mongodb.com/) - this package contains the mongodb database 
2. [PyMongo](http://api.mongodb.com/python/current/) - this packages contains the python driver that will allow us to communicate with the mongodb database.

#### Install

1. Open a command line terminal and execute the following commad to install mongodb.
```console
conda install -c anaconda mongodb
```
2. Open a command line terminal and execute the following commad to install pymongo packge.
```console
conda install -c anaconda pymongo
```

#### Run Mongodb in Windows
1. MongoDB requires a data directory to store all data. MongoDB’s default data directory path is \data\db. Create this folder using the following commands from a Command Prompt:
```console
md \data\db
```

2. To start MongoDB, run mongod.exe. For example, from the Command Prompt:
```console
"C:\Program Files\MongoDB\Server\3.2\bin\mongod.exe"
```

#### Run Mongodb in Mac
1. MongoDB requires a data directory to store all data. MongoDB’s default data directory path is /data/db. Create this folder using the following commands from a Command Prompt.  Note that we run the command as a super user using the "sudo" command:
```console
sudo mkdir /data/db/
```

2. To start MongoDB, run mongod.exe. For example, from the Command Promp.  Note that we run the command as a super user using the "sudo" command:
```console
sudo mongod --config /usr/local/etc/mongod.conf
```

In [1]:
conda install -c anaconda mongodb

SyntaxError: ignored

In [ ]:
conda install -c anaconda pymongo

In [ ]:
#required imports
import os
import json
import datetime
import pymongo
import pprint
import pandas as pd
import numpy as np
from pymongo import MongoClient
print('Mongo version', pymongo.__version__)

We first need to connect to our locally running MongoDB database (<b>Make sure your database is running on your machine</b>). We will use the MongoClient to connect to a local 'test' database that is running on port 27017 (this is the default port).

In [ ]:
client = MongoClient('localhost', 27017)
db = client.assignment1

After installing necessary modules proceed to import the data into your database.

In [ ]:
# Let's delete any existing collections in our database
db.workplace.drop()
db.address.drop()
db.employee.drop()

# Import our files into our three collections
with open('Data/Employee.json') as f:
    db.employee.insert_many(json.load(f))
with open('Data/Workplace.json') as f:
    db.workplace.insert_many(json.load(f))
with open('Data/Address.json') as f:
    db.address.insert_many(json.load(f))

#### Question 1 (10 Marks)

The address collection contains employee from different ages and interests.  Perform a simple query to list all employees that are less than or equal to 50 and like Cooking.

__NOTE:__ the following shows the structure of an Employee document that will help you construct the query.

In [ ]:
pprint.pprint(client.assignment1.employee.find_one())

In [ ]:
cursor = db.employee.find()
#display results in a dataframe so that we can view results of all employee records in an organized table.
pd.DataFrame(list(cursor))
#Total 100 records

In [ ]:
cursor = db.employee.find({"interests": "Cooking", "age": {"$lte": 50 }})
# The above states conditions to isolate those that have the interest of cooking and are less than or equal to 50 years of age. 
pd.DataFrame(list(cursor))
#Total 16 records which fit this criteria.

#### Question 2  (10 Marks)

Insert a new Employee with the following properties:

* First Name: Jake 
* Last Name: Sample
* Email: jakesample@email.com
* Age: 26
* Interest: Biking, Hiking

Also, this employee works for 'Union Planters Corp' and lives at '573 Wojhas Square, Victoria'.
Verify that the insert succeeded and display the generated employees _id attribute.

__HINT__ An Employee document references a Workplace and Address document

In [ ]:
cursor = db.address.find()
pd.DataFrame(list(cursor)) #Tellins us the total amount of records for the address collection. Total 160 records

In [ ]:
cursor = db.employee.find()
pd.DataFrame(list(cursor)) #Tellins us the total amount of records for the employee collection. Total 100 records

In [ ]:
db.employee.insert_one({
    "firstname": "Jake",
    "lastname": "Sample",
    "email": "jakesample@email.com",
    "age": "26",
    "interests": "Biking, Hiking",
    "address_id": "91b5b7b3-2309-4e8a-8247-cd66d626ef0c",
    "workplace_id": "5345fcb9-6297-4b9f-aa15-cbee8460f28f"
})
cursor = db.employee.find()
pd.DataFrame(list(cursor))

#Inserting necessary details. Now 101 records. for the additional workplace and address I just added that to workplace/address id
#This made sense as there were already columns which indicated details for those fields.

#### Question 3 (10 Marks)

Delete all employees that work for 'Great Plains Energy Inc' and are greater than 46 years old and likes 'Tennis'. Once you delete the employees verify the number of employees deleted.

In [ ]:
cursor = db.employee.find({"interests": "Tennis", "age": {"$gt": 46 }})
# display results in a dataframe so that we can display results in a pretty table
pd.DataFrame(list(cursor)) #Based on output we can see that there are a total of 16 employees that like tennis and are +46 yo.

In [ ]:
cursor = db.workplace.find()
# display results in a dataframe so that we can display results in a pretty table
pd.DataFrame(list(cursor)) 
#Did this to isolate the workplace_id for Great Plains Energy which we can use to delete relevant records with the next step.

In [ ]:
pipeline = [
    {"$lookup": 
     { 
         "from": "workplace",  # execute this pipeline operation on the workplace collection
         "localField": "name",  # lookup the workplace_id field in our employee collection
         "foreignField": "name",  # match out localField (i.e. workplace_id) with the _id field in the workplace collection
         "as": "workplace" # store the resulting information in a field name 'workplace'
     }
    },
    {"$lookup": 
     { 
         "from": "employee",  # execute this pipeline operation on the address collection
         "localField": "employee_id",  # lookup the address_id field in our employee collection
         "foreignField": "employee_id",  # match out localField (i.e. employee_id) with the _id field in the address collection
         "as": "employee"  # store the resulting information in a field name 'address'
     }
    }
]

query_result = list(db.employee.aggregate(pipeline))
query_result

#Create a pipeline to link the collections.

In [ ]:
cursor = db.employee.find({"interests": "Tennis", "age": {"$gt": 46 },"workplace_id":"a32bf18d-e0e5-48f2-a851-aa49c80f9460"})
# display results in a dataframe so that we can display results in a pretty table
pd.DataFrame(list(cursor)) #Further isolation of the relevants records. 
#By combining the conditions for age, interest and workplace_id we can isolate the records we need to delete

In [ ]:
delete_result = db.employee.delete_many({"interests": "Tennis", "age": {"$gt": 46 },"workplace_id":"a32bf18d-e0e5-48f2-a851-aa49c80f9460"})
#Proceed to delete those same records
cursor = db.employee.find()
pd.DataFrame(list(cursor))
#This proves that the deletion worked as we now see a total of 4 less records (97 vs 101, this would change without Jake)

In [ ]:
print("Delete Acknowledged:" + str(delete_result.acknowledged))
print("Delete Count:" + str(delete_result.deleted_count))
#The number of deleted records makes sense based on the output

#### Question 4 (12 Marks)
Add a new field called 'industry' to all employees that work for 'Health Net Inc.'.

__HINT__ All a new field to a document is like updating the document

In [ ]:
cursor = db.employee.find()
pd.DataFrame(list(cursor))

In [ ]:
cursor = db.employee.find({"workplace_id":"a222385c-342c-43ea-adbc-9b487a2ee2be"})
pd.DataFrame(list(cursor))
#Confirming that there are a total of 14 records belonging to Health Net Inc. based on isolating with workplace_id

In [ ]:
update_result = db.employee.update_many(
      { "workplace_id" : "a222385c-342c-43ea-adbc-9b487a2ee2be"},
      { "$set": { "industry" : "Construction" } }
);
print("Update Acknowledged:" + str(update_result.acknowledged))
print("Modified Count:" + str(update_result.modified_count))
#Updating to create a new field ("industry") for any employee working for Health Net Inc. Total of 14 records also looks correct

In [ ]:
cursor = db.employee.find({"workplace_id":"a222385c-342c-43ea-adbc-9b487a2ee2be"})
pd.DataFrame(list(cursor))
#Proof that the update/insertion of a new field was succesful.

#### Question 5 (10 Marks)

Create an aggregate query to count the number of employees for each company and sort the output from largest employee count to lowest employee count.

__NOTE__ you will use a pipeline to achieve the computed result.  You should produce a result similar to the following table (the following table contains fake data)
<table>
    <tr><th></th><th>_id</th><th>count</th></tr>
    <tr><td>0</td><td>[Equity Residential Properties Trust]</td><td>19</td></tr>
    <tr><td>...</td><td>...</td><td>...</td></tr>
    <tr><td>7</td><td>[Bell Microproducts Inc.]</td><td>6</td></tr>
    <tr><td>8</td><td>[Kemet Corp.]</td><td>1</td></tr>
</table>

__HINT__ you should make use of the \\$lookup, \\$group and \\$sort pipeline operations

In [ ]:
pipeline = [
{"$lookup": 
     { 
         "from": "workplace",  # execute this pipeline operation on the workplace collection
         "localField": "workplace_id",  # lookup the workplace_id field in our employee collection
         "foreignField": "_id",  # match out localField (i.e. workplace_id) with the _id field in the workplace collection
         "as": "workplace" # store the resulting information in a field name 'workplace'
     }
    },
    {"$lookup": 
     { 
         "from": "address",  # execute this pipeline operation on the address collection
         "localField": "address_id",  # lookup the address_id field in our employee collection
         "foreignField": "_id",  # match out localField (i.e. address_id) with the _id field in the address collection
         "as": "address"  # store the resulting information in a field name 'address'
     }
    }
]

query_result = list(db.employee.aggregate(pipeline))
query_result

list(db.employee.aggregate(pipeline))

#Creating a pipeline to link collections

In [ ]:
update_result = db.employee.update_many(
      { "workplace_id" : "5345fcb9-6297-4b9f-aa15-cbee8460f28f"},
      { "$set": { "name" : "Union Planters Corp" } }
);

update_result1 = db.employee.update_many(
      { "workplace_id" : "2b87eb84-e5b4-4f2c-9e13-dc3ba20a7f7f"},
      { "$set": { "name" : "Aetna Inc." } }
);

update_result2 = db.employee.update_many(
      { "workplace_id" : "a222385c-342c-43ea-adbc-9b487a2ee2be"},
      { "$set": { "name" : "Health Net Inc." } }
);

update_result3 = db.employee.update_many(
      { "workplace_id" : "b2903a4b-5688-4597-90ed-0f06d944bb6d"},
      { "$set": { "name" : "Kemet Corp." } }
);

update_result4 = db.employee.update_many(
      { "workplace_id" : "b2a2844d-aeab-4602-b74c-01bf3b8e9c78"},
      { "$set": { "name" : "Bell Microproducts Inc." } }
);

update_result5 = db.employee.update_many(
      { "workplace_id" : "b12cd444-e65b-4bc2-8cf6-2dbe854a627b"},
      { "$set": { "name" : "Equity Residential Properties Trust" } }
);

update_result6 = db.employee.update_many(
      { "workplace_id" : "023acbac-b32a-477f-a65a-db11bd7d40f3"},
      { "$set": { "name" : "Equity Office Properties Trust" } }
);

update_result7 = db.employee.update_many(
      { "workplace_id" : "cb795afb-8dc3-482f-b3a0-14229a280afa"},
      { "$set": { "name" : "Hilton Solutions" } }
);

update_result8 = db.employee.update_many(
      { "workplace_id" : "a32bf18d-e0e5-48f2-a851-aa49c80f9460"},
      { "$set": { "name" : "Great Plains Energy Inc." } }
);

update_result9 = db.employee.update_many(
      { "workplace_id" : "50275ad1-8140-4e79-8818-21793e3eb0a3"},
      { "$set": { "name" : "Xcel Bear Inc" } }
);

update_result10 = db.employee.update_many(
      { "workplace_id" : "5a113bed-8697-4272-b27a-9c10a990f2a4"},
      { "$set": { "name" : "Novellus Systems Inc" } }
);

update_result11 = db.employee.update_many(
      { "workplace_id" : "e7494243-89e1-407f-b474-ae2b7e62e5ad"},
      { "$set": { "name" : "Citizens Hawk" } }
);

#Update to create a new columnn in employee db ("name" - name of company) based on workplace id

In [ ]:
pipeline = [
     {"$unwind": "$name"},
     {"$group": {"_id": "$name", "count": {"$sum": 1}}}
]

query_result = list(db.employee.aggregate(pipeline))

# Let's store the results in a dataframe so that we can print the results in a table.
# Let's also rename the columns
df = pd.DataFrame(query_result) 
df.rename(columns={'_id': 'name', 'count': 'Employee Counts'}, inplace=True)
df
#Create a table of employee counts based on which company they work for

In [ ]:
Employee_Count_Descending = df.sort_values(['Employee Counts'], ascending=[0])
Employee_Count_Descending
#Based on the output we can see that the amount of employees working for each company is presented in descending order.
#Could add 4 to Great plains energy due to the earlier removal of records, however the total of 96 records adds up
#Do have the option of removing Jake